In [3]:
from matplotlib.font_manager import FontManager
import subprocess

# 查看所有可用字体
fm = FontManager()
mat_fonts = set(f.name for f in fm.ttflist)

# 查看中文字体
chinese_fonts = [f for f in mat_fonts if 'Hei' in f or 'Song' in f or 'Microsoft' in f]
print(chinese_fonts)

['Microsoft YaHei', 'SimHei', 'FangSong']


In [ ]:
"""
plot_weight_hist.py

绘制 SDFMap 与 EntropySDFMap 的第二层、第三层权重直方图对比。

用法示例:
  python isdf/eval/figs/plot_weight_hist.py \
    --sdfmap-ckpt /path/to/sdfmap_checkpoint.pth \
    --entropy-ckpt /path/to/entropy_checkpoint.pth \
    --out /path/to/output_hist.png

说明:
- 从两个 checkpoint 文件中加载模型权重。
- 提取第二层（mid1）和第三层（cat_layer）的权重张量。
- 绘制四个子图（2x2 grid）：
    - 左上：SDFMap 第二层权重直方图
    - 右上：EntropySDFMap 第二层权重直方图
    - 左下：SDFMap 第三层权重直方图
    - 右下：EntropySDFMap 第三层权重直方图
- 支持自动推断层名（对 EntropySDFMap，层名带 entropy_bottleneck 或 ema_w/ema_b）。
"""
import argparse
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def load_checkpoint_weights(ckpt_path: Path):
    """
    加载 checkpoint 并返回 state_dict。
    支持直接 state_dict 或包含 'model_state_dict' 键的字典。
    """
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
    
    ckpt = torch.load(ckpt_path, map_location='cpu')
    
    # 尝试提取 state_dict（常见格式：直接 dict 或包含 model_state_dict）
    if isinstance(ckpt, dict):
        if 'model_state_dict' in ckpt:
            state_dict = ckpt['model_state_dict']
        else:
            state_dict = ckpt
    else:
        raise ValueError(f"Unknown checkpoint format: {type(ckpt)}")
    
    return state_dict

def extract_layer_weights(state_dict, layer_name_patterns):
    """
    从 state_dict 中提取匹配给定模式的层权重。
    layer_name_patterns: list of str patterns（例如 ['mid1', 'cat_layer']）
    返回 dict: {pattern: tensor}
    """
    results = {}
    for pattern in layer_name_patterns:
        matched = []
        for k, v in state_dict.items():
            # 匹配包含 pattern 且包含 'weight' 的键
            if pattern in k and 'weight' in k:
                matched.append((k, v))
        
        if not matched:
            print(f"Warning: No weights found for pattern '{pattern}'")
            results[pattern] = None
        else:
            # 如果有多个匹配，合并或选择主权重
            # 通常第一个匹配的是主权重，或 ema_w 是编码后的权重
            # 优先选择不含 'entropy_bottleneck' 的（即原始权重或 ema_w）
            main_weight = None
            for k, v in matched:
                if 'entropy_bottleneck' not in k:
                    main_weight = v
                    print(f"  Found '{pattern}' weight in key: {k}, shape: {v.shape}")
                    break
            if main_weight is None:
                # fallback: 取第一个
                main_weight = matched[0][1]
                print(f"  Found '{pattern}' weight in key: {matched[0][0]}, shape: {main_weight.shape}")
            
            results[pattern] = main_weight
    
    return results

def plot_histograms(sdfmap_weights, entropy_weights, layer_names, out_path: Path, bins=50):
    """
    绘制 2x2 子图：第二层和第三层的权重直方图对比。
    layer_names: list of 2 layer patterns, e.g., ['mid1', 'cat_layer']
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Weight Histograms: SDFMap vs EntropySDFMap", fontsize=16, fontweight='bold')
    
    # 定义层标签（用于显示）
    layer_labels = ['Layer 2 (mid1)', 'Layer 3 (cat_layer)']
    
    for i, (layer_name, layer_label) in enumerate(zip(layer_names, layer_labels)):
        sdf_w = sdfmap_weights.get(layer_name)
        ent_w = entropy_weights.get(layer_name)
        
        # 左列：SDFMap
        ax_sdf = axes[i, 0]
        if sdf_w is not None:
            w_flat = sdf_w.detach().cpu().numpy().flatten()
            ax_sdf.hist(w_flat, bins=bins, color='steelblue', alpha=0.7, edgecolor='black')
            ax_sdf.set_title(f"iSDF {layer_label}", fontsize=12, fontweight='bold')
            ax_sdf.set_xlabel("Weight value")
            ax_sdf.set_ylabel("Frequency")
            ax_sdf.grid(axis='y', alpha=0.3)

        else:
            ax_sdf.text(0.5, 0.5, f"No data for {layer_label}",
                       ha='center', va='center', transform=ax_sdf.transAxes, fontsize=12)
            ax_sdf.set_title(f"SDFMap {layer_label} (N/A)", fontsize=12)
        
        # 右列：EntropySDFMap
        ax_ent = axes[i, 1]
        if ent_w is not None:
            w_flat = ent_w.detach().cpu().numpy().flatten()
            ax_ent.hist(w_flat, bins=bins, color='darkorange', alpha=0.7, edgecolor='black')
            ax_ent.set_title(f"EntropySDFMap {layer_label}", fontsize=12, fontweight='bold')
            ax_ent.set_xlabel("Weight value")
            ax_ent.set_ylabel("Frequency")
            ax_ent.grid(axis='y', alpha=0.3)
            mean_val = np.mean(w_flat)
            std_val = np.std(w_flat)

        else:
            ax_ent.text(0.5, 0.5, f"No data for {layer_label}",
                       ha='center', va='center', transform=ax_ent.transAxes, fontsize=12)
            ax_ent.set_title(f"EntropySDFMap {layer_label} (N/A)", fontsize=12)
    
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(str(out_path), dpi=200)
    plt.close()
    print(f"Saved histogram plot to: {out_path}")

def main():
    parser = argparse.ArgumentParser(description="Plot weight histograms for SDFMap and EntropySDFMap checkpoints")
    parser.add_argument("--sdfmap-ckpt", type=str, required=False, help="Path to SDFMap checkpoint (.pth)",default="/home/hjx/iSDF/results/iSDF/12-07-25_18-09-10/checkpoints/step_130.000.pth")
    parser.add_argument("--entropy-ckpt", type=str, required=False, help="Path to EntropySDFMap checkpoint (.pth)",default="/home/hjx/iSDF/results/iSDF/12-07-25_18-49-20_entropy/checkpoints/step_130.000.pth")
    parser.add_argument("--out", type=str, default="weight_histograms.png", help="Output image path")
    parser.add_argument("--bins", type=int, default=50, help="Number of histogram bins")
    args = parser.parse_args()
    
    sdfmap_ckpt = Path(args.sdfmap_ckpt)
    entropy_ckpt = Path(args.entropy_ckpt)
    out_path = Path(args.out)
    
    print("Loading SDFMap checkpoint...")
    sdfmap_state = load_checkpoint_weights(sdfmap_ckpt)
    print(f"  Loaded {len(sdfmap_state)} keys from {sdfmap_ckpt.name}")
    
    print("Loading EntropySDFMap checkpoint...")
    entropy_state = load_checkpoint_weights(entropy_ckpt)
    print(f"  Loaded {len(entropy_state)} keys from {entropy_ckpt.name}")
    
    # 定义要提取的层名称模式（第二层：mid1，第三层：cat_layer）
    layer_patterns = ['mid1', 'cat_layer']
    
    print("\nExtracting weights from SDFMap...")
    sdfmap_weights = extract_layer_weights(sdfmap_state, layer_patterns)
    
    print("\nExtracting weights from EntropySDFMap...")
    entropy_weights = extract_layer_weights(entropy_state, layer_patterns)
    
    print("\nGenerating histogram plots...")
    plot_histograms(sdfmap_weights, entropy_weights, layer_patterns, out_path, bins=args.bins)

if __name__ == "__main__":
    main()